In [1]:
# Imports
import sys
import logging
from datetime import datetime
import pandas as pd
from IPython.display import display

sys.path.insert(0, '../../../LOGOS')
from src import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization, plot_equipment_utilization
# Configure logging in the runner (avoid setting basicConfig inside the module)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import json
from pathlib import Path

cwd = Path.cwd()
benchmark = cwd/'benchmarks'
benchmark_results_file = benchmark/'priority_rules_results.json'

## Load Benchmark results
with open(benchmark_results_file, "r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

In [2]:
def run_case(case_name, file_name, json_path,
             schema_file="outage_schema.json",
             benchmark_data=None):
    """
    Runs scheduling comparison for a given case and file.

    Parameters:
        case_name (str): Case identifier (e.g., 'j60')
        file_name (str): File name (e.g., 'j601_1.sm')
        json_path (str): Path to JSON file for Pert model
        schema_file (str): Path to schema file (default: outage_schema.json)
        benchmark_data (dict): Benchmark dataset for RCPSP comparison

    Returns:
        results_df (pd.DataFrame): LOGOS.CPM results
        data_df (pd.DataFrame): RCPSP benchmark results
    """

    results_sgs = {}
    results_pgs = {}
    results_pgs_pr = {}

    # Load Pert Model
    pert = Pert.from_json_file(json_path, schema_path=schema_file)

    prs = [
        'lf', 'ls', 'ef', 'es', 'duration', 'random',
        'mts', 'mtp', 'grpw', 'grd', 'rr', 'avgrr',
        'maxrr', 'minrr','mehh_8000_b','mehh_3375_b',
        'mehh_1000_b','mehh_125_b','gphh_b'
    ]

    # Compute results with Serial
    for rule in prs:
        out = pert.calculateSerialScheduleWithResources(priority_rule=rule)
        results_sgs[rule] = out['scheduled_duration'] - 2  # remove start/end duration

    sgs = ['first', 'max_use_res_ranked', 'max_use_res_shuffled', 'md_knapsack', 'look_ahead']

    # Compute results with Parallel
    for s in sgs:
        out = pert.calculateScheduleWithResources(sgs=s)
        results_pgs[s] = out['scheduled_duration'] - 2  # remove start/end duration

    for rule in prs:
        out = pert.calculateScheduleWithResources(sgs='max_use_res_ranked', priority_rule=rule)
        results_pgs_pr[rule] = out['scheduled_duration'] - 2  # remove start/end duration


    print('Results from LOGOS.CPM Using Serial Generation Scheme:')
    print('-' * 60)
    results_df_sgs = pd.DataFrame(results_sgs, index=[0])
    display(results_df_sgs)

    print('Results from LOGOS.CPM Using Parallel Generation Scheme:')
    print('-' * 60)
    results_df_pgs = pd.DataFrame(results_pgs, index=[0])
    display(results_df_pgs)

    print('Results from LOGOS.CPM Using Parallel Generation Scheme with Priority Rule:')
    print('-' * 60)
    results_df_pgs_pr = pd.DataFrame(results_pgs_pr, index=[0])
    display(results_df_pgs_pr)

    # RCPSP benchmark comparison
    print('Results from RCPSP')
    print('-' * 60)

    if benchmark_data is None:
        raise ValueError("benchmark_data must be provided")

    data = benchmark_data[case_name][file_name]
    data_df = pd.DataFrame(data, index=[0]).filter(like='serial_forward')
    data_df.columns = data_df.columns.str.replace("_serial_forward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)

    return results_df_sgs, results_df_pgs, data_df

## Scheduling with 30 activities

In [3]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j30',
    file_name='j301_1.sm',
    json_path='j301_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=32 | CPM=40.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 07:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 05:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J10 | start=2026-01-01 07:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 05:00 | end=2026-01-01 13:00 | delay=4.0h
DEBUG:root:Serial SGS: scheduled J9 | start=2026-01-01 07:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J12 | start=2026-01-01 14:00 | end=2026-01-01 16:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 09:00 | end=2026-01-01 15:


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED
t=2026-01-01 00:00
completed=[]
ongoing=['J1']
waiting=['J2', 'J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32']
candidates=['J1']
selected=['J1']
t=2026-01-01 01:00
completed=['J1']
ongoing=['J2']
waiting=['J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32']
candidates=['J2', 'J3', 'J4']
selected=['J2']
t=2026-01-

DEBUG:root:t=2026-01-01 23:00 | iter=13 | completed=9/32 | ongoing=4 | waiting=19 | candidates=4 | selected=1 | heap_size=13
DEBUG:root:t=2026-01-02 00:00 | iter=14 | completed=10/32 | ongoing=4 | waiting=18 | candidates=4 | selected=1 | heap_size=12
DEBUG:root:t=2026-01-02 01:00 | iter=15 | completed=12/32 | ongoing=3 | waiting=17 | candidates=5 | selected=1 | heap_size=10
DEBUG:root:t=2026-01-02 02:00 | iter=16 | completed=12/32 | ongoing=4 | waiting=16 | candidates=4 | selected=1 | heap_size=10
DEBUG:root:t=2026-01-02 03:00 | iter=17 | completed=13/32 | ongoing=4 | waiting=15 | candidates=4 | selected=1 | heap_size=10
DEBUG:root:t=2026-01-02 04:00 | iter=18 | completed=14/32 | ongoing=4 | waiting=14 | candidates=4 | selected=1 | heap_size=10
DEBUG:root:t=2026-01-02 05:00 | iter=19 | completed=14/32 | ongoing=5 | waiting=13 | candidates=3 | selected=1 | heap_size=10
DEBUG:root:t=2026-01-02 08:00 | iter=20 | completed=16/32 | ongoing=4 | waiting=12 | candidates=4 | selected=1 | heap_s

t=2026-01-01 23:00
completed=['J1', 'J2', 'J3', 'J4', 'J7', 'J5', 'J6', 'J9', 'J8']
ongoing=['J10', 'J11', 'J13', 'J12']
waiting=['J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32']
candidates=['J12', 'J15', 'J19', 'J27']
selected=['J12']
t=2026-01-02 00:00
completed=['J1', 'J2', 'J3', 'J4', 'J7', 'J5', 'J6', 'J9', 'J8', 'J10']
ongoing=['J11', 'J13', 'J12', 'J15']
waiting=['J14', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32']
candidates=['J15', 'J16', 'J19', 'J27']
selected=['J15']
t=2026-01-02 01:00
completed=['J1', 'J2', 'J3', 'J4', 'J7', 'J5', 'J6', 'J9', 'J8', 'J10', 'J13', 'J12']
ongoing=['J11', 'J15', 'J14']
waiting=['J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32']
candidates=['J14', 'J16', 'J18', 'J19', 'J27']
selected=['J14']
t=2026-01-02 02:00
completed=['J1

DEBUG:root:t=2026-01-02 02:00 | iter=16 | completed=11/32 | ongoing=4 | waiting=17 | candidates=0 | selected=0 | heap_size=10
DEBUG:root:t=2026-01-02 03:00 | iter=17 | completed=13/32 | ongoing=3 | waiting=16 | candidates=1 | selected=1 | heap_size=9
DEBUG:root:t=2026-01-02 04:00 | iter=18 | completed=14/32 | ongoing=2 | waiting=16 | candidates=1 | selected=0 | heap_size=8
DEBUG:root:t=2026-01-02 05:00 | iter=19 | completed=15/32 | ongoing=2 | waiting=15 | candidates=1 | selected=1 | heap_size=7
DEBUG:root:t=2026-01-02 07:00 | iter=20 | completed=16/32 | ongoing=3 | waiting=13 | candidates=3 | selected=2 | heap_size=8
DEBUG:root:t=2026-01-02 08:00 | iter=21 | completed=16/32 | ongoing=3 | waiting=13 | candidates=1 | selected=0 | heap_size=7
DEBUG:root:t=2026-01-02 09:00 | iter=22 | completed=17/32 | ongoing=3 | waiting=12 | candidates=2 | selected=1 | heap_size=7
DEBUG:root:t=2026-01-02 10:00 | iter=23 | completed=19/32 | ongoing=2 | waiting=11 | candidates=3 | selected=1 | heap_size=5

...

,first,max_use_res_ranked,max_use_res_shuffled,md_knapsack,look_ahead
0,50.0,43.0,61.0,61.0,43.0


Results from LOGOS.CPM Using Parallel Generation Scheme with Priority Rule:
------------------------------------------------------------


,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,43.0,46.0,51.0,61.0,45.0,61.0,43.0,61.0,61.0,53.0,61.0,51.0,51.0,61.0,61.0,51.0,61.0,53.0,61.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,51,60,46,49,57,49,49,61,60,53,52,53,52,46,74



## Scheduling with 60 activities

In [4]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j60',
    file_name='j601_1.sm',
    json_path='j601_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=62 | CPM=79.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 11:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 11:00 | end=2026-01-01 20:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J9 | start=2026-01-01 20:00 | end=2026-01-01 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 02:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 21:00 | end=2026-01-02 03:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J14 | start=2026-01-01 02:00 | end=2026-01-01 04:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 01:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J18 | start=2026-01-02 03:00 | end=2026-01-02 13:


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J59 | start=2026-01-04 03:00 | end=2026-01-04 06:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J61 | start=2026-01-03 08:00 | end=2026-01-03 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J57 | start=2026-01-03 21:00 | end=2026-01-04 03:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J56 | start=2026-01-03 00:00 | end=2026-01-03 08:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J55 | start=2026-01-03 11:00 | end=2026-01-03 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J60 | start=2026-01-02 11:00 | end=2026-01-02 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J53 | start=2026-01-02 23:00 | end=2026-01-03 00:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J51 | start=2026-01-02 07:00 | end=2026-01-02 11:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J54 | start=2026-01-03 07:00 | end=2026-01-03 11:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J58 | start=2026-01-02 04:00 | end=2026-01-02 14:00 | delay=0.0h
DEBUG:root:Serial SG

t=2026-01-01 00:00
completed=[]
ongoing=['J1']
waiting=['J2', 'J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35', 'J36', 'J37', 'J38', 'J39', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J48', 'J49', 'J50', 'J51', 'J52', 'J53', 'J54', 'J55', 'J56', 'J57', 'J58', 'J59', 'J60', 'J61', 'J62']
candidates=['J1']
selected=['J1']
t=2026-01-01 01:00
completed=['J1']
ongoing=['J2']
waiting=['J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35', 'J36', 'J37', 'J38', 'J39', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J48', 'J49', 'J50', 'J51', 'J52', 'J53', 'J54', 'J55', 'J56', 'J57', 'J58', 'J59', 'J60', 'J61', 'J62']
candidates=['J2', 

DEBUG:root:t=2026-01-03 05:00 | iter=39 | completed=47/62 | ongoing=4 | waiting=11 | candidates=1 | selected=1 | heap_size=11
DEBUG:root:t=2026-01-03 06:00 | iter=40 | completed=48/62 | ongoing=4 | waiting=10 | candidates=1 | selected=1 | heap_size=11
DEBUG:root:t=2026-01-03 07:00 | iter=41 | completed=49/62 | ongoing=4 | waiting=9 | candidates=1 | selected=1 | heap_size=10
DEBUG:root:t=2026-01-03 08:00 | iter=42 | completed=50/62 | ongoing=3 | waiting=9 | candidates=0 | selected=0 | heap_size=8
DEBUG:root:t=2026-01-03 09:00 | iter=43 | completed=51/62 | ongoing=3 | waiting=8 | candidates=1 | selected=1 | heap_size=8
DEBUG:root:t=2026-01-03 10:00 | iter=44 | completed=52/62 | ongoing=3 | waiting=7 | candidates=1 | selected=1 | heap_size=8
DEBUG:root:t=2026-01-03 11:00 | iter=45 | completed=52/62 | ongoing=3 | waiting=7 | candidates=0 | selected=0 | heap_size=7
DEBUG:root:t=2026-01-03 13:00 | iter=46 | completed=54/62 | ongoing=2 | waiting=6 | candidates=1 | selected=1 | heap_size=6
DEB

...

,first,max_use_res_ranked,max_use_res_shuffled,md_knapsack,look_ahead
0,92.0,86.0,84.0,92.0,86.0


Results from LOGOS.CPM Using Parallel Generation Scheme with Priority Rule:
------------------------------------------------------------


,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,86.0,86.0,85.0,85.0,85.0,85.0,82.0,92.0,92.0,92.0,84.0,86.0,86.0,84.0,86.0,84.0,86.0,92.0,96.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,86,88,77,77,121,80,77,106,84,98,77,85,77,109,121


## Scheduling with 90 activities

In [5]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j90',
    file_name='j901_1.sm',
    json_path='j901_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=92 | CPM=69.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 11:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 01:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 02:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J15 | start=2026-01-01 11:00 | end=2026-01-01 18:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J20 | start=2026-01-01 18:00 | end=2026-01-01 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-01 09:00 | end=2026-01-01 17:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J12 | start=2026-01-01 17:00 | end=2026-01-01 19:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J5 | start=2026-01-01 02:00 | end=2026-01-01 04


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J83 | start=2026-01-03 11:00 | end=2026-01-03 20:00 | delay=3.0h
DEBUG:root:Serial SGS: scheduled J32 | start=2026-01-03 01:00 | end=2026-01-03 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J50 | start=2026-01-03 04:00 | end=2026-01-03 12:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J57 | start=2026-01-03 04:00 | end=2026-01-03 09:00 | delay=3.0h
DEBUG:root:Serial SGS: scheduled J73 | start=2026-01-03 08:00 | end=2026-01-03 13:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J85 | start=2026-01-02 22:00 | end=2026-01-03 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J80 | start=2026-01-03 07:00 | end=2026-01-03 08:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J61 | start=2026-01-03 10:00 | end=2026-01-03 13:00 | delay=4.0h
DEBUG:root:Serial SGS: scheduled J82 | start=2026-01-03 08:00 | end=2026-01-03 18:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J51 | start=2026-01-03 09:00 | end=2026-01-03 17:00 | delay=4.0h
DEBUG:root:Serial SG

t=2026-01-01 00:00
completed=[]
ongoing=['J1']
waiting=['J2', 'J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35', 'J36', 'J37', 'J38', 'J39', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J48', 'J49', 'J50', 'J51', 'J52', 'J53', 'J54', 'J55', 'J56', 'J57', 'J58', 'J59', 'J60', 'J61', 'J62', 'J63', 'J64', 'J65', 'J66', 'J67', 'J68', 'J69', 'J70', 'J71', 'J72', 'J73', 'J74', 'J75', 'J76', 'J77', 'J78', 'J79', 'J80', 'J81', 'J82', 'J83', 'J84', 'J85', 'J86', 'J87', 'J88', 'J89', 'J90', 'J91', 'J92']
candidates=['J1']
selected=['J1']
t=2026-01-01 01:00
completed=['J1']
ongoing=['J2']
waiting=['J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35

DEBUG:root:t=2026-01-02 00:00 | iter=17 | completed=25/92 | ongoing=8 | waiting=59 | candidates=7 | selected=1 | heap_size=28
DEBUG:root:t=2026-01-02 01:00 | iter=18 | completed=25/92 | ongoing=8 | waiting=59 | candidates=6 | selected=0 | heap_size=27
DEBUG:root:t=2026-01-02 02:00 | iter=19 | completed=27/92 | ongoing=8 | waiting=57 | candidates=8 | selected=2 | heap_size=26
DEBUG:root:t=2026-01-02 03:00 | iter=20 | completed=30/92 | ongoing=9 | waiting=53 | candidates=11 | selected=4 | heap_size=27
DEBUG:root:t=2026-01-02 04:00 | iter=21 | completed=30/92 | ongoing=9 | waiting=53 | candidates=7 | selected=0 | heap_size=26
DEBUG:root:t=2026-01-02 06:00 | iter=22 | completed=31/92 | ongoing=8 | waiting=53 | candidates=7 | selected=0 | heap_size=25
DEBUG:root:t=2026-01-02 07:00 | iter=23 | completed=35/92 | ongoing=7 | waiting=50 | candidates=14 | selected=3 | heap_size=23
DEBUG:root:t=2026-01-02 08:00 | iter=24 | completed=35/92 | ongoing=7 | waiting=50 | candidates=11 | selected=0 | he

...

,first,max_use_res_ranked,max_use_res_shuffled,md_knapsack,look_ahead
0,277.0,86.0,89.0,97.0,94.0


Results from LOGOS.CPM Using Parallel Generation Scheme with Priority Rule:
------------------------------------------------------------


,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,81.0,81.0,90.0,85.0,97.0,96.0,86.0,96.0,99.0,93.0,94.0,89.0,89.0,94.0,81.0,84.0,81.0,93.0,97.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,94,98,83,82,111,88,89,101,95,108,84,101,84,102,148


## Scheduling with 120 activities

In [6]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j120',
    file_name='j1201_1.sm',
    json_path='j1201_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=122 | CPM=101.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J6 | start=2026-01-01 05:00 | end=2026-01-01 08:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J7 | start=2026-01-01 08:00 | end=2026-01-01 18:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-01 18:00 | end=2026-01-02 00:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 03:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 03:00 | end=2026-01-01 10:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J18 | start=2026-01-02 00:00 | end=2026-01-02 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 03:00 | end=2026-01-01 0


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J28 | start=2026-01-04 07:00 | end=2026-01-04 16:00 | delay=12.0h
DEBUG:root:Serial SGS: scheduled J33 | start=2026-01-03 16:00 | end=2026-01-04 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J36 | start=2026-01-04 05:00 | end=2026-01-04 14:00 | delay=4.0h
DEBUG:root:Serial SGS: scheduled J38 | start=2026-01-02 21:00 | end=2026-01-03 06:00 | delay=37.0h
DEBUG:root:Serial SGS: scheduled J53 | start=2026-01-03 01:00 | end=2026-01-03 10:00 | delay=37.0h
DEBUG:root:Serial SGS: scheduled J54 | start=2026-01-03 19:00 | end=2026-01-04 04:00 | delay=9.0h
DEBUG:root:Serial SGS: scheduled J59 | start=2026-01-04 16:00 | end=2026-01-05 01:00 | delay=55.0h
DEBUG:root:Serial SGS: scheduled J82 | start=2026-01-02 21:00 | end=2026-01-03 06:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J86 | start=2026-01-04 23:00 | end=2026-01-05 08:00 | delay=38.0h
DEBUG:root:Serial SGS: scheduled J115 | start=2026-01-04 23:00 | end=2026-01-05 08:00 | delay=13.0h
DEBUG:root:Se

t=2026-01-01 00:00
completed=[]
ongoing=['J1']
waiting=['J2', 'J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35', 'J36', 'J37', 'J38', 'J39', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J48', 'J49', 'J50', 'J51', 'J52', 'J53', 'J54', 'J55', 'J56', 'J57', 'J58', 'J59', 'J60', 'J61', 'J62', 'J63', 'J64', 'J65', 'J66', 'J67', 'J68', 'J69', 'J70', 'J71', 'J72', 'J73', 'J74', 'J75', 'J76', 'J77', 'J78', 'J79', 'J80', 'J81', 'J82', 'J83', 'J84', 'J85', 'J86', 'J87', 'J88', 'J89', 'J90', 'J91', 'J92', 'J93', 'J94', 'J95', 'J96', 'J97', 'J98', 'J99', 'J100', 'J101', 'J102', 'J103', 'J104', 'J105', 'J106', 'J107', 'J108', 'J109', 'J110', 'J111', 'J112', 'J113', 'J114', 'J115', 'J116', 'J117', 'J118', 'J119', 'J120', 'J121', 'J122']
candidates=['J1']
selected=['J1']
t=2026-01-01 01:00
completed=['J1']
ongoing=['J2

DEBUG:root:t=2026-01-02 15:00 | iter=35 | completed=41/122 | ongoing=7 | waiting=74 | candidates=16 | selected=4 | heap_size=34
DEBUG:root:t=2026-01-02 16:00 | iter=36 | completed=43/122 | ongoing=8 | waiting=71 | candidates=15 | selected=3 | heap_size=34
DEBUG:root:t=2026-01-02 17:00 | iter=37 | completed=44/122 | ongoing=9 | waiting=69 | candidates=13 | selected=2 | heap_size=35
DEBUG:root:t=2026-01-02 18:00 | iter=38 | completed=44/122 | ongoing=9 | waiting=69 | candidates=11 | selected=0 | heap_size=34
DEBUG:root:t=2026-01-02 19:00 | iter=39 | completed=45/122 | ongoing=10 | waiting=67 | candidates=12 | selected=2 | heap_size=35
DEBUG:root:t=2026-01-02 20:00 | iter=40 | completed=45/122 | ongoing=10 | waiting=67 | candidates=10 | selected=0 | heap_size=34
DEBUG:root:t=2026-01-02 21:00 | iter=41 | completed=46/122 | ongoing=10 | waiting=66 | candidates=11 | selected=1 | heap_size=33
DEBUG:root:t=2026-01-02 22:00 | iter=42 | completed=50/122 | ongoing=8 | waiting=64 | candidates=15 |

...

,first,max_use_res_ranked,max_use_res_shuffled,md_knapsack,look_ahead
0,345.0,120.0,148.0,148.0,120.0


Results from LOGOS.CPM Using Parallel Generation Scheme with Priority Rule:
------------------------------------------------------------


,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,126.0,120.0,128.0,127.0,132.0,126.0,121.0,139.0,126.0,133.0,121.0,133.0,133.0,121.0,124.0,121.0,129.0,130.0,178.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,132,144,119,123,147,123,125,153,140,156,124,138,114,157,193
